# YOLO26m 균열 세그멘테이션 학습

Roboflow 데이터셋을 내려받아 YOLO26m-seg를 1024 해상도로 학습하고 결과를 Google Drive에 저장하는 Colab 노트북입니다. 위에서부터 순서대로 실행하세요.

## 1. 라이브러리 설치

In [ ]:
!pip install -U ultralytics roboflow -q

## 2. Google Drive 연결

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
PROJECT_DIR = DRIVE_ROOT / 'YOLO26_Crack_Final'
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
print('결과 저장 위치:', PROJECT_DIR)

## 3. Roboflow 데이터셋 다운로드

API 키는 노트북에 저장하지 않고 실행할 때 입력합니다.

In [ ]:
from getpass import getpass
from roboflow import Roboflow

ROBOFLOW_API_KEY = getpass('Roboflow API key: ')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)
project = rf.workspace('aos-workspace').project('erc2-59n5c')
version = project.version(1)
dataset = version.download('yolo26')
print('데이터셋 위치:', dataset.location)

## 4. 데이터셋 및 GPU 확인

In [ ]:
import torch
import yaml

DATASET_DIR = Path(dataset.location)
DATA_YAML = DATASET_DIR / 'data.yaml'
if not DATA_YAML.exists():
    raise FileNotFoundError(f'data.yaml을 찾지 못했습니다: {DATA_YAML}')

with open(DATA_YAML, 'r', encoding='utf-8') as file:
    print(yaml.safe_load(file))

if not torch.cuda.is_available():
    raise RuntimeError('Colab 런타임에서 GPU를 선택하세요.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
for split in ['train', 'valid', 'val', 'test']:
    image_dir = DATASET_DIR / split / 'images'
    label_dir = DATASET_DIR / split / 'labels'
    if image_dir.exists():
        image_count = sum(p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS for p in image_dir.iterdir())
        label_count = len(list(label_dir.glob('*.txt'))) if label_dir.exists() else 0
        print(f'{split:5s} | 이미지: {image_count:4d}개 | 라벨: {label_count:4d}개')

## 5. YOLO26m 학습

최대 250 epoch까지 학습하며, 검증 성능이 50 epoch 동안 개선되지 않으면 조기 종료합니다.

In [ ]:
from ultralytics import YOLO

RUN_NAME = 'yolo26m_seg_1024_patience50'
model = YOLO('yolo26m-seg.pt')
train_results = model.train(
    data=str(DATA_YAML),
    epochs=250,
    patience=50,
    imgsz=1024,
    batch=1,
    device=0,
    pretrained=True,
    optimizer='auto',
    amp=True,
    workers=2,
    cache=False,
    close_mosaic=15,
    project=str(PROJECT_DIR),
    name=RUN_NAME,
    exist_ok=False,
    save=True,
    save_period=50,
    plots=True,
    seed=42,
    deterministic=True,
)
print('학습 결과:', train_results.save_dir)

## 6. 최적 모델 검증

In [ ]:
RUN_DIR = PROJECT_DIR / RUN_NAME
BEST_MODEL = RUN_DIR / 'weights' / 'best.pt'
if not BEST_MODEL.exists():
    raise FileNotFoundError(f'best.pt를 찾지 못했습니다: {BEST_MODEL}')

best_model = YOLO(str(BEST_MODEL))
metrics = best_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=1024,
    batch=1,
    device=0,
    plots=True,
    project=str(PROJECT_DIR),
    name=f'{RUN_NAME}_validation',
    exist_ok=True,
)
print(f'Mask Precision: {metrics.seg.mp:.4f}')
print(f'Mask Recall: {metrics.seg.mr:.4f}')
print(f'Mask mAP50: {metrics.seg.map50:.4f}')
print(f'Mask mAP50-95: {metrics.seg.map:.4f}')

## 7. 학습 결과 확인

In [ ]:
import pandas as pd
from IPython.display import Image, display

RESULTS_CSV = RUN_DIR / 'results.csv'
if RESULTS_CSV.exists():
    results_df = pd.read_csv(RESULTS_CSV)
    results_df.columns = results_df.columns.str.strip()
    print('진행된 epoch:', len(results_df))
    display(results_df.tail())

RESULTS_PNG = RUN_DIR / 'results.png'
if RESULTS_PNG.exists():
    display(Image(filename=str(RESULTS_PNG), width=1200))

## 주의사항

- Roboflow API 키를 코드 셀이나 GitHub에 저장하지 마세요.
- 이미 같은 이름의 결과 폴더가 있으면 `RUN_NAME`을 변경하거나 기존 폴더를 별도로 백업하세요.
- Colab 런타임이 중단되어도 Google Drive에 저장된 `last.pt`에서 학습을 재개할 수 있습니다.